# Phase 4 · Submission S1 — Architecture A (Decoder-only Transformer) + LORMA, trained with LP-FT

**Task:** Court Judgment Prediction (CJPE) — predict the binary outcome (0/1) of a case from its judgment text. **Primary metric:** Macro-F1.

**Why this version.** Three earlier runs of A + LORMA trained the classifier and LORMA together from a random classifier and collapsed to one prediction for every case, while a logistic regression on *frozen* A features reached 0.55–0.58 dev-clean Macro-F1 (`P4_diagnostic.ipynb`). The likely reason: with a random head, the first LORMA updates (learning rate 5e-4) amplify A's few huge residual dimensions and wreck the features before the head finds a signal. This version therefore uses **LP-FT** (linear probing, then fine-tuning; Kumar et al., 2022): first fit the classifier on frozen features, then fine-tune LORMA + classifier gently from that starting point.

**Method**
1. **Backbone:** Phase-1 Architecture A (12-layer decoder, 125.0M parameters), all Phase-1 weights frozen; dropout off.
2. **Adaptation — LORMA (Bihany, Patel & Modi, 2025), compulsory in this phase:** q, k, v, out and fc1, fc2 of all 12 layers get a multiplicative low-rank update `W_eff = (I + (α/r)·B·A)·W0` (r = 8, α = 16). `B = 0` at start, so the model begins exactly as Phase 1.
3. **Long documents:** lower-cased text; the **last 2 chunks of 510 BPE tokens** of each case (the outcome signal sits at the end of the judgment). Each chunk → [mean of token states ; final `[EOS]` state] (1,536 numbers); the case vector is the average over its chunks, **standardised per dimension** with training-set statistics (and clipped to ±10) → linear layer over 2 classes.
4. **Step 1 — linear probe (LP):** case vectors of all 31.6k training cases from the frozen Phase-1 model → class-balanced logistic regression (C chosen on dev-clean) → its weights initialise the classifier. This model is scored as "epoch 0".
5. **Step 2 — fine-tuning (FT):** LORMA + classifier trained with class-weighted cross-entropy, small learning rates (1e-4), 4 epochs, mixed precision, both T4 GPUs. The epoch (0–4) with the best dev-clean Macro-F1 is kept, so fine-tuning can only be kept if it helps.
6. **Model selection without leakage:** all 1,517 test cases are also in `dev.json`; they are **never** used. C, epoch, threshold and section setting are chosen on the 1,988 **dev-clean** cases. Test labels are never read; train only.
7. **Section analysis:** first 2 vs last 2 / 4 / 8 chunks on dev-clean; the best is used for test.

**Before running (Kaggle)**
- Accelerator: **GPU T4 ×2** · Add-ons → Secrets: **`HF`** · Internet on.
- Add Input → datasets: **`rohitbatra2004/phase111`** (Architecture A) and **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer). Optional: a previous output of P4_S1 (its `tokens_cache.pt` is reused — saves ≈ 10 min).
- Expected time ≈ 2.5 hours (tokens ≈ 10 min, frozen features ≈ 10 min, probe ≈ 5 min, ≈ 23 min per epoch). Checkpoints every 500 steps and every epoch; features are cached; to resume, add this notebook's previous output as an input and run again.

Outputs: `/kaggle/working/P4_S1_A_LORMA/` — upload `submission.zip`.

In [ ]:
# Cell 1 — Imports, random seed and device.
# Every later cell uses the names defined here.
import os, json, glob, math, time, random, shutil, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import hashlib
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE, "| GPUs visible:", torch.cuda.device_count())

In [ ]:
# Cell 2 — Paths and hyperparameters for this submission.
# Every setting lives here; no other cell hard-codes a value.
RUN_NAME = "P4_S1_A_LORMA"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
CKPT_DIR = WORK_DIR / "checkpoints"
OUT_DIR = WORK_DIR / "outputs"
for folder in (WORK_DIR, DATA_DIR, CKPT_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"

# Long-document input: chunks of 510 BPE tokens (+ [BOS], [EOS] = 512).
CHUNK_TOKENS = 510
TRAIN_CHUNKS = 2                 # last 2 chunks of every case for the probe and for training
MAX_TAIL_CHUNKS = 8              # up to 8 chunks from the end are stored for the section analysis
HEAD_CHUNKS = 2                  # first 2 chunks are stored for the "beginning of the judgment" comparison
TAIL_CHARS = 60000               # only the last/first characters are tokenised (enough for 8 / 2 chunks)
HEAD_CHARS = 12000

# LORMA on attention and feed-forward projections of all 12 layers.
LORMA_RANK = 8
LORMA_ALPHA = 16.0
LORMA_TARGETS_ATTN = ("q_proj", "k_proj", "v_proj", "out_proj")
LORMA_TARGETS_FFN = ("fc1", "fc2")

# Case vector and classifier.
FEATURE_CLAMP = 10.0             # standardised features are clipped to ±10 (guards against near-constant dimensions)
DROPOUT = 0.1

# Step 1 — linear probe on frozen Phase-1 features.
PROBE_C_VALUES = (0.01, 0.1, 1.0)   # inverse regularisation strengths tried; the best on dev-clean is used
FEATURE_BATCH_DOCS = 16

# Step 2 — LP-FT fine-tuning (small learning rates so the probe's good start is not destroyed).
EPOCHS = 4                       # 4 epochs: dev-clean 0.6542 → 0.6750 → 0.6947 → 0.6901 (best = epoch 3)
BATCH_DOCS = 8                   # cases per forward pass (2 chunks each → 16 sequences of 512 tokens)
GRAD_ACCUM = 2                   # effective batch = 16 cases
LR_ADAPTER = 1e-4
LR_HEAD = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
GRAD_CLIP = 1.0
SAVE_EVERY_STEPS = 500
CHECK_EVERY_STEPS = 500          # quick dev check on 600 dev-clean cases
CHECK_CASES = 600
EVAL_BATCH_DOCS = 16
USE_AMP = DEVICE.type == "cuda"
print("Run:", RUN_NAME, "| outputs:", WORK_DIR)

In [ ]:
# Cell 3 — Download Phase_4_CJPE from Hugging Face using the Kaggle secret "HF".
from kaggle_secrets import UserSecretsClient
snapshot_download(repo_id=HF_REPO_ID, repo_type="dataset", token=UserSecretsClient().get_secret("HF"),
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_4_CJPE/**"])
P4_DIR = DATA_DIR / "Phase_4_CJPE"
for name in ("Train/train.json", "Train/dev.json", "Test/test.json"):
    assert (P4_DIR / name).is_file(), f"Missing {name}"
print("Phase 4 data ready in", P4_DIR)

In [ ]:
# Cell 4 — Load the splits, lower-case and normalise the text, remove duplicate training cases,
# and build "dev-clean" = dev cases whose ids are NOT in the test set. Test labels are never read.
def normalise(text):
    return " ".join(text.lower().split())

def text_key(text):
    return hashlib.md5(text.encode()).hexdigest()

raw_train = json.load(open(P4_DIR / "Train/train.json"))
raw_dev = json.load(open(P4_DIR / "Train/dev.json"))
raw_test = json.load(open(P4_DIR / "Test/test.json"))

train_cases, seen = [], set()
for d in raw_train:
    text = normalise(d["text"])
    key = text_key(text)
    if key in seen:
        continue
    seen.add(key)
    train_cases.append({"id": str(d["id"]), "text": text, "label": int(d["label"])})

test_cases = [{"id": str(d["id"]), "text": normalise(d["text"])} for d in raw_test]       # labels deliberately ignored
test_ids = {c["id"] for c in test_cases}
dev_all = [{"id": str(d["id"]), "text": normalise(d["text"]), "label": int(d["label"])} for d in raw_dev]
dev_cases = [c for c in dev_all if c["id"] not in test_ids]
del raw_train, raw_dev, raw_test

print(f"train: {len(train_cases):,} unique cases (duplicates removed) | labels {Counter(c['label'] for c in train_cases)}")
print(f"dev  : {len(dev_all):,} cases, of which {len(dev_all) - len(dev_cases):,} are also in test and are excluded "
      f"→ dev-clean {len(dev_cases):,} | labels {Counter(c['label'] for c in dev_cases)}")
print(f"test : {len(test_cases):,} cases")

In [ ]:
# Cell 5 — Tokenise the beginning and the end of every case with the Phase-1 BPE tokenizer (cached).
# Stored per case: up to the last 8×510 tokens ("tail") and the first 2×510 tokens ("head").
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
PAD_ID, BOS_ID, EOS_ID = (tokenizer.token_to_id(t) for t in ("[PAD]", "[BOS]", "[EOS]"))
assert (PAD_ID, BOS_ID, EOS_ID) == (0, 2, 3)
TAIL_LEN, HEAD_LEN = MAX_TAIL_CHUNKS * CHUNK_TOKENS, HEAD_CHUNKS * CHUNK_TOKENS

def encode_cases(cases):
    tails, heads = [], []
    for start in range(0, len(cases), 512):
        batch = cases[start:start + 512]
        tail_enc = tokenizer.encode_batch([c["text"][-TAIL_CHARS:] for c in batch])
        head_enc = tokenizer.encode_batch([c["text"][:HEAD_CHARS] for c in batch])
        for c, t, h in zip(batch, tail_enc, head_enc):
            ids = t.ids[1:] if len(c["text"]) > TAIL_CHARS else t.ids      # drop a possibly cut first token
            tails.append(np.array(ids[-TAIL_LEN:], dtype=np.uint16))
            heads.append(np.array(h.ids[:HEAD_LEN], dtype=np.uint16))
    return tails, heads

cache = WORK_DIR / "tokens_cache.pt"
if not cache.exists():
    previous = [p for p in glob.glob(f"/kaggle/input/**/{RUN_NAME}/tokens_cache.pt", recursive=True)]
    if previous:
        shutil.copy(previous[0], cache)
if cache.exists():
    tokens = torch.load(cache, weights_only=False)
    print("Loaded cached tokens from", cache)
else:
    start_time = time.time()
    tokens = {name: encode_cases(cases) for name, cases in (("train", train_cases), ("dev", dev_cases), ("test", test_cases))}
    torch.save(tokens, cache)
    print(f"Tokenised all cases in {(time.time() - start_time) / 60:.1f} min → {cache}")
for name, cases in (("train", train_cases), ("dev", dev_cases), ("test", test_cases)):
    assert len(tokens[name][0]) == len(cases)
    lengths = np.array([len(t) for t in tokens[name][0]])
    print(f"{name}: tail tokens kept median {int(np.median(lengths))}, "
          f"cases shorter than 2 chunks {100 * np.mean(lengths < 2 * CHUNK_TOKENS):.1f}%")

In [ ]:
# Cell 6 — Phase-1 Architecture A classes and the pretrained weights, verified against the Phase-1 metadata.
# The weights stay in memory so that build_model() can create a fresh adapted copy.
a_paths = sorted(glob.glob("/kaggle/input/**/architecture_A/model_A_state_dict.pt", recursive=True))
assert a_paths, "Add the dataset rohitbatra2004/phase111 as an input."
A_ROOT = Path(a_paths[0]).parent
A_METADATA = json.load(open(A_ROOT / "model_A_metadata.json"))
A_CONFIG = A_METADATA["model_config"]
assert A_CONFIG["pre_norm"] and A_CONFIG["tie_embeddings"], "Phase-1 A is a pre-LN decoder with tied embeddings"

class CausalSelfAttention(nn.Module):
    """Same parameters as Phase 1; uses PyTorch's fused causal attention for speed and memory."""
    def __init__(self, cfg):
        super().__init__()
        d = cfg["d_model"]
        self.n_heads, self.dropout = cfg["n_heads"], cfg["dropout"]
        self.q_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.k_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.v_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.out_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.resid_dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        B, T, C = x.shape
        split = lambda t: t.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        out = F.scaled_dot_product_attention(split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x)),
                                             dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.resid_dropout(self.out_proj(out.transpose(1, 2).reshape(B, T, C)))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["use_bias"])
        self.fc2 = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["use_bias"])
        self.dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg["d_model"])
        self.attention = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg["d_model"])
        self.feed_forward = FeedForward(cfg)

    def forward(self, x):
        x = x + self.attention(self.ln_1(x))
        return x + self.feed_forward(self.ln_2(x))

class ArchitectureABackbone(nn.Module):
    """Phase-1 decoder-only LM without the vocabulary projection.
    Returns the final-normalised state of every token: [batch, tokens, d_model]."""
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = nn.Embedding(cfg["context_length"], cfg["d_model"])
        self.embedding_dropout = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = nn.LayerNorm(cfg["d_model"])
        # Tied output weight, kept so that the saved Phase-1 state dict loads strictly; unused here.
        self.output_weight = self.token_embedding.weight

    def forward(self, input_ids):
        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        x = self.embedding_dropout(self.token_embedding(input_ids) + self.position_embedding(positions))
        for block in self.blocks:
            x = block(x)
        return self.final_norm(x)

saved = torch.load(a_paths[0], map_location="cpu", weights_only=True)
# The export is either the raw state dict or a dict that wraps it under "state_dict".
PHASE1_STATE = saved["state_dict"] if "state_dict" in saved else saved
del saved

check = ArchitectureABackbone(A_CONFIG)
check.load_state_dict(PHASE1_STATE, strict=True)
n_backbone = sum(p.numel() for p in check.parameters())
assert n_backbone == int(A_METADATA["parameter_count"]), "Parameter count differs from Phase-1 metadata"
del check
print(json.dumps(A_CONFIG, indent=2))
print(f"Phase-1 Architecture A weights verified from {A_ROOT} — {n_backbone:,} parameters")

In [ ]:
# Cell 7 — LORMA: multiplicative low-rank update of a frozen linear layer, W_eff = (I + (α/r)·B·A)·W0.
# Applied to q/k/v/out and fc1/fc2 in all 12 layers; B = 0 at start, so the model initially equals Phase 1.
class LoRMALinear(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.lorma_A = nn.Parameter(torch.empty(rank, base.out_features))
        self.lorma_B = nn.Parameter(torch.zeros(base.out_features, rank))
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))

    def forward(self, x):
        w0x = F.linear(x, self.base.weight)                               # W0 x (without bias)
        update = F.linear(F.linear(w0x, self.lorma_A), self.lorma_B)       # B (A (W0 x))
        out = w0x + self.scale * update
        return out + self.base.bias if self.base.bias is not None else out

def make_lorma_backbone():
    bb = ArchitectureABackbone(A_CONFIG)
    bb.load_state_dict(PHASE1_STATE, strict=True)
    for p in bb.parameters():
        p.requires_grad_(False)
    for block in bb.blocks:
        for name in LORMA_TARGETS_ATTN:
            setattr(block.attention, name, LoRMALinear(getattr(block.attention, name), LORMA_RANK, LORMA_ALPHA))
        for name in LORMA_TARGETS_FFN:
            setattr(block.feed_forward, name, LoRMALinear(getattr(block.feed_forward, name), LORMA_RANK, LORMA_ALPHA))
    return bb

# Checks on the device: (1) with B = 0 the output equals the frozen Phase-1 model; (2) A and B receive gradients.
probe_ids = torch.randint(4, 1000, (2, 16), device=DEVICE)
plain = ArchitectureABackbone(A_CONFIG); plain.load_state_dict(PHASE1_STATE, strict=True); plain = plain.to(DEVICE).eval()
probe_bb = make_lorma_backbone().to(DEVICE).eval()
with torch.no_grad():
    assert torch.allclose(plain(probe_ids), probe_bb(probe_ids), atol=1e-4), "LORMA with B = 0 must equal Phase 1"
    lorma_layers = [m for m in probe_bb.modules() if isinstance(m, LoRMALinear)]
    for m in lorma_layers:
        m.lorma_B.normal_(0, 1e-3)                                       # temporary non-zero B for the gradient check
probe_bb.train()
probe_bb(probe_ids).float().pow(2).mean().backward()
assert all(m.lorma_A.grad is not None and m.lorma_A.grad.abs().sum() > 0 for m in lorma_layers), "LORMA gets no gradient"
n_lorma = sum(p.numel() for p in probe_bb.parameters() if p.requires_grad)
print(f"LORMA on {len(lorma_layers)} layers — {n_lorma:,} trainable backbone parameters ({100 * n_lorma / n_backbone:.2f}%)")
del plain, probe_bb, lorma_layers
torch.cuda.empty_cache()

In [ ]:
# Cell 8 — Case classifier: average over chunks of [mean token state ; final state] → standardise → linear layer.
# The standardisation statistics and the classifier's starting weights come from the linear probe in Cell 11.
class CaseClassifier(nn.Module):
    def __init__(self, backbone, hidden):
        super().__init__()
        self.backbone = backbone
        self.register_buffer("feat_mean", torch.zeros(2 * hidden))     # set from training features in Cell 11
        self.register_buffer("feat_std", torch.ones(2 * hidden))
        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(2 * hidden, 2)

    def case_features(self, ids, chunk_mask, standardise=True):
        """ids [B, K, T], chunk_mask [B, K] → case vectors [B, 2·hidden] (standardised and clipped, or raw)."""
        B, K, T = ids.shape
        flat_ids, valid = ids.view(B * K, T), chunk_mask.view(-1)
        token_mask = flat_ids[valid].ne(PAD_ID)
        states = self.backbone(flat_ids[valid]).float()
        m = token_mask.unsqueeze(-1).float()
        mean = (states * m).sum(1) / m.sum(1).clamp(min=1.0)
        last = states[torch.arange(states.size(0), device=states.device), token_mask.sum(1) - 1]
        vecs = torch.zeros(B * K, 2 * states.size(-1), device=ids.device)
        vecs[valid] = torch.cat([mean, last], dim=-1)
        w = chunk_mask.float().unsqueeze(-1)
        case = (vecs.view(B, K, -1) * w).sum(1) / w.sum(1).clamp(min=1.0)      # average over the case's chunks
        if not standardise:
            return case
        return ((case - self.feat_mean) / self.feat_std).clamp(-FEATURE_CLAMP, FEATURE_CLAMP)

    def forward(self, ids, chunk_mask, chunk_pos):
        """Returns (logits [B, 2], chunk_mask) — chunk_pos is accepted for a uniform batch interface."""
        return self.classifier(self.dropout(self.case_features(ids, chunk_mask))), chunk_mask

class FeatureRunner(nn.Module):
    """Wraps the model so that DataParallel can compute raw case vectors on both GPUs."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, ids, chunk_mask):
        return self.model.case_features(ids, chunk_mask, standardise=False)

def build_model():
    torch.manual_seed(SEED)
    return CaseClassifier(make_lorma_backbone(), A_CONFIG["d_model"]).to(DEVICE)

model = build_model()
TRAINABLE_NAMES = [n for n, p in model.named_parameters() if p.requires_grad]
SAVED_NAMES = TRAINABLE_NAMES + ["feat_mean", "feat_std"]     # what checkpoints store
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
N_GPUS = torch.cuda.device_count()
train_model = nn.DataParallel(model) if N_GPUS > 1 else model          # both T4s share each batch when available
print(f"Trainable parameters: {n_trainable:,} of {n_total:,} ({100 * n_trainable / n_total:.2f}%) | GPUs used: {max(N_GPUS, 1)}")

In [ ]:
# Cell 9 — Turn stored tokens into model inputs ("tail" = last chunks, "head" = first) and class weights.
def case_chunks(index, split, part, n_chunks):
    tail, head = tokens[split][0][index], tokens[split][1][index]
    if part == "tail":
        ids = tail[-n_chunks * CHUNK_TOKENS:]
        pieces = [ids[max(0, end - CHUNK_TOKENS):end] for end in range(len(ids), 0, -CHUNK_TOKENS)][::-1]
    else:
        ids = head[:n_chunks * CHUNK_TOKENS]
        pieces = [ids[s:s + CHUNK_TOKENS] for s in range(0, len(ids), CHUNK_TOKENS)]
    pieces = [p for p in pieces if len(p) > 0] or [np.array([], dtype=np.uint16)]
    return [[BOS_ID] + p.tolist() + [EOS_ID] for p in pieces]

def make_batch(indices, split, part, n_chunks):
    per_case = [case_chunks(i, split, part, n_chunks) for i in indices]
    K = max(len(c) for c in per_case)
    T = max(len(x) for c in per_case for x in c)
    ids = torch.full((len(indices), K, T), PAD_ID, dtype=torch.long)
    mask = torch.zeros(len(indices), K, dtype=torch.bool)
    pos = torch.zeros(len(indices), K, dtype=torch.long)
    for b, chunks in enumerate(per_case):
        for k, x in enumerate(chunks):
            ids[b, k, :len(x)] = torch.tensor(x)
            mask[b, k] = True
            pos[b, k] = (len(chunks) - 1 - k) if part == "tail" else min(k, MAX_TAIL_CHUNKS - 1)
    return ids.to(DEVICE), mask.to(DEVICE), pos.to(DEVICE)

train_labels = torch.tensor([c["label"] for c in train_cases])
counts = torch.bincount(train_labels, minlength=2).float()
CLASS_WEIGHTS = (counts.sum() / (2 * counts)).to(DEVICE)               # balanced weights for the 59/41 train split
ids, mask, pos = make_batch([0, 1], "train", "tail", TRAIN_CHUNKS)
print("Example batch:", tuple(ids.shape), "| class weights:", CLASS_WEIGHTS.tolist())


In [ ]:
# Cell 10 — Prediction and evaluation: P(label = 1) for every case, Macro-F1 at 0.5 and at the best threshold.
@torch.no_grad()
def predict_probs(model, split, part="tail", n_chunks=TRAIN_CHUNKS, limit=None):
    model.eval()
    n = len(tokens[split][0]) if limit is None else min(limit, len(tokens[split][0]))
    probs = []
    for start in range(0, n, EVAL_BATCH_DOCS):
        batch = make_batch(list(range(start, min(start + EVAL_BATCH_DOCS, n))), split, part, n_chunks)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits, _ = model(*batch)
        probs.append(torch.softmax(logits.float(), -1)[:, 1].cpu())
    return torch.cat(probs).numpy()

def scores_at(gold, probs, threshold):
    pred = (probs >= threshold).astype(int)
    return {"macro_f1": f1_score(gold, pred, average="macro", zero_division=0),
            "macro_p": precision_score(gold, pred, average="macro", zero_division=0),
            "macro_r": recall_score(gold, pred, average="macro", zero_division=0), "threshold": float(threshold)}

def evaluate(model, part="tail", n_chunks=TRAIN_CHUNKS, limit=None):
    probs = predict_probs(model, "dev", part, n_chunks, limit)
    gold = np.array([c["label"] for c in dev_cases])[:len(probs)]
    at_half = scores_at(gold, probs, 0.5)
    best = max((scores_at(gold, probs, t) for t in np.arange(0.20, 0.81, 0.01)), key=lambda s: s["macro_f1"])
    return {"at_0.5": at_half, "tuned": best}, probs

print("Evaluation helpers ready (dev-clean only).")

In [ ]:
# Cell 11 — Step 1, linear probe (the "LP" of LP-FT): case vectors of every training and dev-clean case from the
# frozen Phase-1 model (LORMA B = 0) → class-balanced logistic regression → its weights initialise the classifier.
from sklearn.linear_model import LogisticRegression

def trainable_state():
    full = model.state_dict()
    return {n: full[n].detach().cpu() for n in SAVED_NAMES}

LATEST, BEST = CKPT_DIR / "latest.pt", CKPT_DIR / "best.pt"

@torch.no_grad()
def raw_case_features(split, n_chunks=TRAIN_CHUNKS):
    runner = FeatureRunner(model)
    runner = nn.DataParallel(runner) if N_GPUS > 1 else runner
    runner.eval()
    n, out = len(tokens[split][0]), []
    for start in range(0, n, FEATURE_BATCH_DOCS):
        ids, mask, _ = make_batch(list(range(start, min(start + FEATURE_BATCH_DOCS, n))), split, "tail", n_chunks)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out.append(runner(ids, mask).float().cpu())
    return torch.cat(out)

feature_cache = WORK_DIR / "frozen_features.pt"
if not feature_cache.exists():
    previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/frozen_features.pt", recursive=True))
    if previous:
        shutil.copy(previous[0], feature_cache)
if feature_cache.exists():
    frozen = torch.load(feature_cache)
    print("Loaded cached frozen features from", feature_cache)
else:
    start_time = time.time()
    frozen = {"train": raw_case_features("train"), "dev": raw_case_features("dev")}
    torch.save(frozen, feature_cache)
    print(f"Frozen Phase-1 features of {len(frozen['train']):,} train + {len(frozen['dev']):,} dev-clean cases "
          f"in {(time.time() - start_time) / 60:.1f} min")
assert frozen["train"].shape[0] == len(train_cases) and frozen["dev"].shape[0] == len(dev_cases)

feat_mean = frozen["train"].mean(0)
feat_std = frozen["train"].std(0).clamp(min=1e-4)
def standardise(x):
    return ((x - feat_mean) / feat_std).clamp(-FEATURE_CLAMP, FEATURE_CLAMP).numpy()
X_train, X_dev = standardise(frozen["train"]), standardise(frozen["dev"])
y_train, y_dev = train_labels.numpy(), np.array([c["label"] for c in dev_cases])

probe_results, best_probe = {}, None
for c_value in PROBE_C_VALUES:
    start_time = time.time()
    clf = LogisticRegression(C=c_value, max_iter=2000, class_weight="balanced").fit(X_train, y_train)
    p_dev = clf.predict_proba(X_dev)[:, 1]
    f1 = max(f1_score(y_dev, (p_dev >= t).astype(int), average="macro") for t in np.arange(0.20, 0.81, 0.01))
    probe_results[str(c_value)] = {"dev_clean_macro_f1": round(float(f1), 4), "spread": round(float(p_dev.std()), 4)}
    print(f"  logistic regression C = {c_value}: dev-clean Macro-F1 {f1:.4f} | spread {p_dev.std():.3f} "
          f"| {time.time() - start_time:.0f} s")
    if best_probe is None or f1 > best_probe[0]:
        best_probe = (f1, c_value, clf)
PROBE_F1, PROBE_C, probe = best_probe

# Copy the probe into the model: softmax over [−z/2, +z/2] equals the logistic regression's sigmoid(z).
with torch.no_grad():
    model.feat_mean.copy_(feat_mean); model.feat_std.copy_(feat_std)
    w = torch.tensor(probe.coef_[0], dtype=torch.float32); b = float(probe.intercept_[0])
    model.classifier.weight.copy_(torch.stack([-w / 2, w / 2])); model.classifier.bias.copy_(torch.tensor([-b / 2, b / 2]))
probe_dev, probe_dev_p = evaluate(model)
print(f"Probe (C = {PROBE_C}) copied into the model → epoch 0: dev-clean Macro-F1 {probe_dev['tuned']['macro_f1']:.4f} "
      f"at threshold {probe_dev['tuned']['threshold']:.2f} (logistic regression itself {PROBE_F1:.4f}) | "
      f"spread {probe_dev_p.std():.3f}")
assert abs(probe_dev["tuned"]["macro_f1"] - PROBE_F1) < 0.02, "The model does not reproduce the probe"
torch.save({"trainable": trainable_state(), "epoch": 0, "dev": probe_dev}, BEST)       # epoch 0 = linear probe only
del X_train, X_dev

In [ ]:
# Cell 12 — Step 2 (the "FT" of LP-FT): optimiser, learning-rate schedule and checkpoint resume.
adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and "lorma_" in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "lorma_" not in n]   # classifier
optimizer = torch.optim.AdamW([{"params": adapter_params, "lr": LR_ADAPTER}, {"params": head_params, "lr": LR_HEAD}],
                              weight_decay=WEIGHT_DECAY)
steps_per_epoch = math.ceil(len(train_cases) / BATCH_DOCS)
total_updates = EPOCHS * math.ceil(steps_per_epoch / GRAD_ACCUM)
warmup_updates = int(WARMUP_FRACTION * total_updates)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda u: (u + 1) / max(warmup_updates, 1) if u < warmup_updates
                                              else max(0.0, (total_updates - u) / max(total_updates - warmup_updates, 1)))
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

if not LATEST.exists():
    # Resume only from a run of THIS version (same saved parameter names); older S1 versions are ignored.
    for previous in sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/checkpoints/latest.pt", recursive=True)):
        if set(torch.load(previous, map_location="cpu", weights_only=False)["trainable"]) == set(SAVED_NAMES):
            shutil.copy(previous, LATEST)
            if Path(previous).with_name("best.pt").exists():
                shutil.copy(Path(previous).with_name("best.pt"), BEST)
            print("Copied checkpoints from", Path(previous).parent)
            break
        print("Ignoring checkpoints of an older S1 version in", Path(previous).parent)

state = {"epoch": 0, "step": 0, "best_f1": probe_dev["tuned"]["macro_f1"], "best_epoch": 0,   # epoch 0 = probe
         "history": [], "train_seconds": 0.0}
if LATEST.exists():
    ckpt = torch.load(LATEST, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["trainable"], strict=False)
    optimizer.load_state_dict(ckpt["optimizer"]); scheduler.load_state_dict(ckpt["scheduler"]); scaler.load_state_dict(ckpt["scaler"])
    state = ckpt["state"]
    print(f"Resumed at epoch {state['epoch'] + 1}, step {state['step']} (best dev-clean Macro-F1 {state['best_f1']:.4f})")
else:
    print("No checkpoint found — fine-tuning starts from the Phase-1 weights + the probe classifier.")
print(f"{steps_per_epoch} steps per epoch, {total_updates} optimiser updates in total")

In [ ]:
# Cell 13 — Fine-tuning LORMA + classifier: last 2 chunks per case, class-weighted cross-entropy, mixed precision, gradient accumulation.
# The frozen backbone runs in eval mode (no dropout): dropping one of its huge outlier dimensions rescales whole token
# vectors through the final LayerNorm, which would add noise during training that never occurs at test time.
# Every epoch ends with a dev-clean evaluation; best.pt keeps the best of epoch 0 (probe), 1, …, EPOCHS.
def save_latest():
    torch.save({"trainable": trainable_state(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "state": state}, LATEST)

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
while state["epoch"] < EPOCHS:
    epoch = state["epoch"]
    order = list(range(len(train_cases)))
    random.Random(SEED + epoch).shuffle(order)                         # same order after a resume
    train_model.train()
    model.backbone.eval()      # frozen Phase-1 network runs without dropout (only the adapter/head train)
    start_time, loss_sum, loss_count = time.time(), 0.0, 0
    for step in range(state["step"], steps_per_epoch):
        batch_idx = order[step * BATCH_DOCS:(step + 1) * BATCH_DOCS]
        ids, mask, pos = make_batch(batch_idx, "train", "tail", TRAIN_CHUNKS)
        labels = train_labels[batch_idx].to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits, _ = train_model(ids, mask, pos)
        loss = F.cross_entropy(logits.float(), labels, weight=CLASS_WEIGHTS) / GRAD_ACCUM
        scaler.scale(loss).backward()
        loss_sum, loss_count = loss_sum + loss.item() * GRAD_ACCUM, loss_count + 1
        if (step + 1) % GRAD_ACCUM == 0 or step + 1 == steps_per_epoch:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(adapter_params + head_params, GRAD_CLIP)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        state["step"] = step + 1
        if state["step"] % SAVE_EVERY_STEPS == 0:
            state["train_seconds"] += time.time() - start_time; start_time = time.time()
            save_latest()
            print(f"  epoch {epoch + 1} step {state['step']}/{steps_per_epoch} | mean loss {loss_sum / loss_count:.4f}", flush=True)
        if state["step"] % CHECK_EVERY_STEPS == 0:
            quick, quick_p = evaluate(model, limit=CHECK_CASES)
            print(f"    quick dev check ({CHECK_CASES} cases): Macro-F1 {quick['tuned']['macro_f1']:.4f} | "
                  f"spread of P(label=1) {quick_p.std():.3f}", flush=True)
            train_model.train()
            model.backbone.eval()
    state["train_seconds"] += time.time() - start_time
    dev, dev_p = evaluate(model)
    record = {"epoch": epoch + 1, "train_loss": loss_sum / max(loss_count, 1), "dev_prob_std": float(dev_p.std()),
              **{f"dev_{k}": v for k, v in dev.items()}}
    state["history"].append(record)
    mark = ""
    if dev["tuned"]["macro_f1"] > state["best_f1"]:
        state.update(best_f1=dev["tuned"]["macro_f1"], best_epoch=epoch + 1)
        torch.save({"trainable": trainable_state(), "epoch": epoch + 1, "dev": dev}, BEST)
        mark = "  <- best"
    state["epoch"], state["step"] = epoch + 1, 0
    save_latest()
    print(f"Epoch {epoch + 1} | loss {record['train_loss']:.4f} | dev-clean Macro-F1 {dev['at_0.5']['macro_f1']:.4f} at 0.5, "
          f"{dev['tuned']['macro_f1']:.4f} at threshold {dev['tuned']['threshold']:.2f} | "
          f"spread of P(label=1) {dev_p.std():.3f}{mark}")
    if dev_p.std() < 0.01:
        print("  WARNING: almost identical predictions for every case — the model has not found a signal yet.")
print(f"Best dev-clean Macro-F1 {state['best_f1']:.4f} at epoch {state['best_epoch']} | "
      f"training time {state['train_seconds'] / 60:.1f} min")

In [ ]:
# Cell 14 — Best epoch: dev-clean report, confusion matrix, and what the false positives / negatives look like.
best = torch.load(BEST, map_location="cpu", weights_only=False)
model.load_state_dict(best["trainable"], strict=False)
dev_result, dev_probs = evaluate(model)
THRESHOLD = dev_result["tuned"]["threshold"]
gold = np.array([c["label"] for c in dev_cases])
pred = (dev_probs >= THRESHOLD).astype(int)
print(f"Best epoch {best['epoch']} — dev-clean Macro-F1 {dev_result['tuned']['macro_f1']:.4f} at threshold {THRESHOLD:.2f} "
      f"(P {dev_result['tuned']['macro_p']:.4f}, R {dev_result['tuned']['macro_r']:.4f}); "
      f"{dev_result['at_0.5']['macro_f1']:.4f} at 0.5")
print(classification_report(gold, pred, digits=4, zero_division=0))
cm = confusion_matrix(gold, pred)
print("Confusion matrix [rows = gold 0/1, columns = predicted 0/1]:", cm.tolist())

lengths = np.array([len(c["text"].split()) for c in dev_cases])
groups = {"true negatives": (gold == 0) & (pred == 0), "false positives": (gold == 0) & (pred == 1),
          "false negatives": (gold == 1) & (pred == 0), "true positives": (gold == 1) & (pred == 1)}
error_stats = {name: {"count": int(g.sum()), "median_words": int(np.median(lengths[g])) if g.any() else 0,
                      "mean_confidence": float(np.mean(np.abs(dev_probs[g] - 0.5) + 0.5)) if g.any() else 0.0}
               for name, g in groups.items()}
for name, s in error_stats.items():
    print(f"  {name:16s}: {s['count']:4d} cases, median length {s['median_words']:5d} words, mean confidence {s['mean_confidence']:.3f}")
json.dump({"dev": dev_result, "best_epoch": best["epoch"], "history": state["history"], "confusion_matrix": cm.tolist(),
           "error_stats": error_stats}, open(OUT_DIR / "dev_metrics.json", "w"), indent=2)

In [ ]:
# Cell 15 — Section analysis on dev-clean: first 2 chunks vs last 2 / 4 / 8 chunks (same trained model).
# The best setting on dev-clean (with its own tuned threshold) is used for the test predictions.
section_results = {}
for part, n_chunks in (("head", 2), ("tail", 2), ("tail", 4), ("tail", 8)):
    result, _ = evaluate(model, part, n_chunks)
    section_results[f"{'first' if part == 'head' else 'last'}_{n_chunks}"] = {"part": part, "n_chunks": n_chunks, **result}
    print(f"  {('first' if part == 'head' else 'last')} {n_chunks} chunks: Macro-F1 {result['tuned']['macro_f1']:.4f} "
          f"(threshold {result['tuned']['threshold']:.2f}), {result['at_0.5']['macro_f1']:.4f} at 0.5")
best_section = max(section_results, key=lambda k: section_results[k]["tuned"]["macro_f1"])
TEST_PART, TEST_CHUNKS = section_results[best_section]["part"], section_results[best_section]["n_chunks"]
TEST_THRESHOLD = section_results[best_section]["tuned"]["threshold"]
print(f"Chosen for test: {best_section} (threshold {TEST_THRESHOLD:.2f})")
json.dump(section_results, open(OUT_DIR / "section_analysis.json", "w"), indent=2)

In [ ]:
# Cell 16 — Predict the test cases, validate the format and build submission.zip.
# prediction.json is a JSON list of {"id": ..., "pred": 0 or 1}, one entry per test case, in test-file order.
test_probs = predict_probs(model, "test", TEST_PART, TEST_CHUNKS)
prediction = [{"id": c["id"], "pred": int(p >= TEST_THRESHOLD)} for c, p in zip(test_cases, test_probs)]
assert [p["id"] for p in prediction] == [c["id"] for c in test_cases] and len(set(test_ids)) == len(prediction)
assert all(p["pred"] in (0, 1) and isinstance(p["pred"], int) for p in prediction)
json.dump(prediction, open(WORK_DIR / "prediction.json", "w"), indent=2)
with zipfile.ZipFile(WORK_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(WORK_DIR / "prediction.json", arcname="prediction.json")
with zipfile.ZipFile(WORK_DIR / "submission.zip") as zf:
    assert zf.namelist() == ["prediction.json"]
print(f"submission.zip: {len(prediction)} cases | predicted label counts {dict(Counter(p['pred'] for p in prediction))}")

In [ ]:
# Cell 17 — Run summary for the README, the experiment log and results/submissions.csv.
summary = {
    "run": RUN_NAME, "backbone": "Architecture A — decoder-only Transformer (Phase 1)",
    "technique": "LORMA (attention + FFN), LP-FT: linear probe then fine-tuning", "lorma_rank": LORMA_RANK,
    "probe": {"chosen_C": PROBE_C, "dev_clean_macro_f1": PROBE_F1, "all": probe_results},
    "trainable_params": n_trainable, "total_params": n_total, "lorma_params": n_lorma,
    "train_cases": len(train_cases), "dev_clean_cases": len(dev_cases), "epochs": state["epoch"],
    "best_epoch": state["best_epoch"], "train_minutes": round(state["train_seconds"] / 60, 1),
    "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE.type == "cuda" else None,
    "dev_clean": dev_result, "sections": {k: v["tuned"]["macro_f1"] for k, v in section_results.items()},
    "test_setting": {"part": TEST_PART, "chunks": TEST_CHUNKS, "threshold": TEST_THRESHOLD},
}
json.dump(summary, open(OUT_DIR / "run_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))